### Instruct Models & the Chat Template

Here the model *follows instructions* and *holds a conversation* instead of merely autocompleting.

1. **Instruct models.** A base model is trained only to predict the next token over raw text. An instruct model is that base model fine-tuned on examples of following instructions and chatting (supervised fine-tuning, then preference tuning). Uses `.generate()` — it just *behaves* the way people expect from a chatbot.
2. **The chat template.** Chat models are fed a *conversation* as a list of `{"role", "content"}` messages


#### Setup

We're on Transformers v5 (PyTorch-only). A GPU runtime is nice but not required — this model is tiny enough to run on CPU.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


#### The model: `HuggingFaceTB/SmolLM2-360M-Instruct`

https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct

We're using a deliberately **tiny** instruct model (360M parameters) from HuggingFaceTB — Hugging Face's own research team.
- It's **small** (~720 MB), so it downloads in seconds and runs anywhere. Perfect for a live session.
- It's **ungated** — no license to accept, no access token needed.


At 360M parameters this model is for *learning the mechanics*, not for impressive answers.

In [2]:
checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

model

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960, padding_idx=2)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (r

#### The messages format

A conversation is a list of dictionaries, each with a **role** and its **content**. Three roles cover almost everything:

- `system` — sets the assistant's overall behaviour/persona (optional).
- `user` — what the person says.
- `assistant` — what the model says back.

In [3]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is the capital of France?'}]

#### `apply_chat_template`

The model doesn't actually consume that list of dictionaries — under the hood it still reads one flat string of tokens. Each chat model ships a **chat template** (a small Jinja template inside its tokenizer) that describes exactly how to flatten the messages into the string it was trained on, including the special delimiter tokens that mark where each turn begins and ends.

tokenize=False` let's us *see* that string instead of token IDs. We also pass `add_generation_prompt=True`, which appends the tokens that say "the assistant speaks next" — the cue that makes the model *reply* rather than continue the user's sentence.

In [4]:
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



Those `<|im_start|>` / `<|im_end|>` markers are **ChatML**, the template family SmolLM2 (and Qwen) use. Notice how each turn is wrapped, and how the string ends with `<|im_start|>assistant` — that trailing opener is what `add_generation_prompt=True` added, telling the model it's now the assistant's turn to speak.

A different model (Llama, Mistral, Gemma) uses completely different delimiters, but `apply_chat_template` reads each model's own template, so the *same* messages list works everywhere.

#### Generating a reply


In [5]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are a helpful assistant.
user
What is the capital of France?
assistant
The capital of France is Paris.


Notice the decoded output contains the **whole** conversation, including our prompt — because `generate()` returns the input tokens followed by the new ones. Usually we only want the assistant's fresh reply, so we slice off the input tokens (everything up to the length of `inputs`) before decoding.

In [6]:
reply_ids = outputs[0][inputs["input_ids"].shape[-1]:]

print(reply_ids)
print(tokenizer.decode(reply_ids, skip_special_tokens=True))

tensor([ 504, 3575,  282, 4649,  314, 7042,   30,    2])
The capital of France is Paris.


#### Wrapping it in a helper



In [7]:
def chat(messages, max_new_tokens=120):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
    )

    reply_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(reply_ids, skip_special_tokens=True)

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Give me three ideas for a weekend project."},
]

print(chat(messages))

1. DIY Woodworking Project: Create a beautiful wooden birdhouse or a custom picture frame. You can use a combination of wood, nails, and screws to build it.

2. DIY Garden Project: Create a garden with a variety of flowers, herbs, and vegetables. You can use a combination of seeds, soil, and gardening tools to create a thriving garden.

3. DIY Water Feature: Build a small water feature like a birdbath or a small fountain. You can use a combination of materials like plastic, metal, and wood to create a beautiful and functional water feature.


#### Steering behaviour with the system prompt

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Always answer in exactly one short sentence."},
    {"role": "user", "content": "What is a computer?"},
]

print(chat(messages))

A computer is an electronic device that can perform calculations and store information.


#### Multi-turn conversation

This is *why* the messages format exists. To continue a conversation, you append the assistant's reply to the list and add the next user turn, then send the whole history back. The model sees the full context, so follow-ups like "and another one" make sense.

In [10]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Suggest a name for a pet cat."},
]

# First reply
reply = chat(conversation)
print("Assistant:", reply)

# Append the reply, then ask a follow-up that relies on context
conversation.append({"role": "assistant", "content": reply})
conversation.append({"role": "user", "content": "Now suggest one for a dog."})

print("Assistant:", chat(conversation))

Assistant: 1. Whisker Whisper - Whisker Whisper is a playful and affectionate cat that loves to play and snuggle.
2. Whisker Wanda - Whisker Wanda is a playful and friendly cat that loves to play and snuggle.
3. Whisker Whiskers - Whisker Whiskers is a playful and affectionate cat that loves to play and snuggle.
4. Whisker Whiskers - Whisker Whiskers is a playful and affectionate cat that loves to play and snuggle
Assistant: 1. Whisker Wanda - Whisker Wanda is a playful and friendly dog that loves to play and snuggle.
2. Whisker Whiskers - Whisker Whiskers is a playful and affectionate dog that loves to play and snuggle.
3. Whisker Whiskers - Whisker Whiskers is a playful and affectionate dog that loves to play and snuggle.
4. Whisker Whiskers - Whisker Whiskers is a playful and affectionate dog that loves to play and snuggle


#### The pipeline shortcut


In [11]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model=checkpoint,
    device=device
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

output = chatbot(messages, max_new_tokens=100)

print(output[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The capital of France is Paris.


#### Classic tasks, done by a chat model

In Transformers v5 the dedicated `summarization`, `translation`, and `question-answering` pipelines were removed — the recommended way to do those now is to **ask a chat model**. With the pattern above, that's just a well-worded user message.

In [12]:
passage = (
    "The Hugging Face Hub hosts hundreds of thousands of models, datasets, and demos. "
    "It lets people download pretrained models, share their own, and try them in the browser. "
    "Its libraries, like Transformers and Datasets, have become standard tools for building NLP applications."
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Summarize the following in one sentence:\n\n{passage}"},
]

print(chat(messages))

The Hugging Face Hub hosts a vast collection of pretrained models, datasets, and demos, allowing users to download, share, and use them in web applications.
